# GNSS Paper 1 - overnight GPU pipeline (generalization + defense)

Runs `run_gpu_experiments.py`, which SMOKE-TESTS then runs the two heavy experiments:

- `23_generalization.py` (full, all 13 models) -> `generalization.csv`
- `24_defense.py` (diagnostic adversarial-training baseline, 6 DL) -> `defense_baseline.csv`

The pipeline smoke-tests each stage first (a few minutes) so any error surfaces early, removes the smoke files, then runs the full stages. ~2-3 h total on a P100.

## Run it overnight WITHOUT losing the result
Use **Save Version -> Save & Run All (Commit)** (top-right), NOT the interactive Run. Commit mode runs headless in the background and stores everything in `/kaggle/working/` as that version's **Output**, downloadable for good after the session ends. The last cell also PRINTS both CSVs as text as a backup. You can close your laptop.

**Before you commit, set in the right panel:** Accelerator = **GPU**, Internet = **On**, and **Add Input** = your dataset with `texbat_track_combined.csv`.

## 1. Clone the code (Paper-1 branch)

In [ ]:
import os, subprocess, sys
REPO   = "https://github.com/Ojerinde/Master_Research.git"
BRANCH = "paper1-experiment"
DST    = "/kaggle/working/repo"
# Private repo? REPO = "https://<GITHUB_TOKEN>@github.com/Ojerinde/Master_Research.git"
if not os.path.exists(DST):
    subprocess.run(["git","clone","--depth","1","-b",BRANCH,REPO,DST], check=True)
os.chdir(DST)
print("cwd:", os.getcwd()); print("top-level:", sorted(os.listdir("."))[:20])

## 2. Place the corpus CSV where the loader expects it

In [ ]:
import glob, shutil, os
src = glob.glob("/kaggle/input/**/texbat_track_combined.csv", recursive=True)
assert src, "Attach the Kaggle dataset that contains texbat_track_combined.csv (right panel > Add Input)."
os.makedirs("data/processed", exist_ok=True)
shutil.copy(src[0], "data/processed/texbat_track_combined.csv")
print(f"CSV placed: {os.path.getsize('data/processed/texbat_track_combined.csv'):,} bytes")

## 3. Environment check (do NOT `pip install -r requirements.txt` here)

In [ ]:
import sys, subprocess, importlib, torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY"))
if not torch.cuda.is_available():
    print("
[WARN] No GPU detected. Set Accelerator = GPU in the right panel, then re-run.")
for mod, pip_name in [("imblearn","imbalanced-learn"),("xgboost","xgboost"),("lightgbm","lightgbm"),("sklearn","scikit-learn")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pip_name], check=True)
print("deps OK")

## 4. Run the full pipeline (smoke -> full generalization -> full defense)

In [ ]:
import os, subprocess, sys, time
env = dict(os.environ, PYTHONPATH=".", PYTHONWARNINGS="ignore")
t0 = time.time()
p = subprocess.Popen([sys.executable, "-u", "run_gpu_experiments.py", "--with-defense"],
                     env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
p.wait()
print(f"
[pipeline done] exit={p.returncode}  elapsed={(time.time()-t0)/60:.1f} min")
assert p.returncode == 0, f"pipeline failed with exit {p.returncode} (scroll up for the traceback)."

## 5. Save + PRINT both result tables (recoverable even if a file is lost)

In [ ]:
import shutil, os, pandas as pd
pd.set_option('display.max_rows', None); pd.set_option('display.width', 200)
for name in ['generalization.csv', 'defense_baseline.csv']:
    srcp = f'results/tables/{name}'
    if not os.path.exists(srcp):
        print(f'[missing] {srcp}'); continue
    dst = f'/kaggle/working/{name}'; shutil.copy(srcp, dst)
    g = pd.read_csv(dst)
    print('='*70); print(f'{name}  rows={len(g)}'); print('='*70)
    print(g.round(4).to_string(index=False))
    print(f'
----BEGIN {name}----'); print(g.to_csv(index=False)); print(f'----END {name}----
')
print('Both saved to /kaggle/working/ -> download from the committed version Output tab.')

### After it finishes
Download `generalization.csv` and `defense_baseline.csv` from the committed version's **Output** tab (persist indefinitely), or copy the text between the `BEGIN/END` markers. Send both back.
- generalization: must have **both** `classical` and `deep` rows; ds2 stays the collapse case.
- defense: expect adversarial training to lift recall under FGSM/PGD only partially, with the detector still falling at eps=0.2 (partial protection, not a fix).